# Dataspace PULL Transfer Flow

This notebook demonstrates the execution of the Dataspace Protocol (DSP 2025-1) transfer process in PULL mode over Eunomia RPC (`/dsp/current/transfers/rpc/setup-*`):
1. Executes prerequisites and contract negotiation (`contract_negotiation.ipynb` with `setup_catalog_policies_oauth.ipynb`).
2. Initiates PULL transfer request (`setup-request`).
3. Starts data transfer from Provider (`setup-start`).
4. Suspends transfer from Provider (`setup-suspension`).
5. Resumes transfer from Provider (`setup-start`).
6. Enforces protocol role constraint (rejecting Consumer-initiated start).
7. Enforces JSON-LD RDF shape validation (rejecting non-RDF / plain JSON).
8. Terminates transfer from Consumer (`setup-termination`).
9. Queries final transfer process state via the Transfer Agent Management API.

## 1. Prerequisites
Execute catalog configuration with secret management and complete DSP 2025-1 Contract Negotiation to obtain an active contract agreement ID.

In [ ]:
%run ./setup_catalog_policies_oauth.ipynb
%run ./contract_negotiation.ipynb

## 2. Transfer Request Setup (Consumer -> Provider)
The Consumer initiates the PULL transfer by sending a `RpcTransferRequestMessage` via `/dsp/current/transfers/rpc/setup-request` referencing the contract agreement ID.
The Provider creates a transfer process in `REQUESTED` state and returns its `providerPid`.

In [11]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["transferAgentModel"]["correlation"]["consumerPid"]
transfer_process_provider_pid = response_as_json["transferAgentModel"]["correlation"]["providerPid"]
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "agreementId": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "format": "http+asd",
    "providerAddress": "http://127.0.0.1:1200/dsp/current"
  },
  "response": {
    "@conte

## 3. Start Transfer (Provider -> Consumer)
According to DSP 2025-1, in a PULL transfer the Provider sends the `TransferStartMessage` to the Consumer, transmitting the data endpoint from which the Consumer can pull data.
The Provider invokes `/dsp/current/transfers/rpc/setup-start`, transitioning the transfer state from `REQUESTED` to `STARTED`.

In [12]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
    "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:353c621b-3f5b-4420-aaaf-ed777380f088",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
      "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
      "agreementId": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
      "call

## 4. Suspend Transfer (Provider -> Consumer)
The Provider temporarily suspends data access by sending a `RpcTransferSuspensionMessage` via `/dsp/current/transfers/rpc/setup-suspension`.
The transfer process transitions from `STARTED` to `SUSPENDED`.

In [13]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "code": "SUSPEND",
    "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
    "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:353c621b-3f5b-4420-aaaf-ed777380f088",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "SUSPENDED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": [
        "Suspending for demonstration purposes"
      ],
      "code": "SUSPEND"
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",

## 5. Resume Transfer (Provider -> Consumer)
In DSP 2025-1, a suspended transfer is resumed by sending another `TransferStartMessage`. Under DSP role rules, only the Provider sends start messages in PULL mode.
The Provider invokes `/dsp/current/transfers/rpc/setup-start`, transitioning the transfer process back from `SUSPENDED` to `STARTED`.

In [14]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
    "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:353c621b-3f5b-4420-aaaf-ed777380f088",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
      "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
      "agreementId": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
      "call

## 6. Role Validation Test: Reject Consumer-Initiated Start
DSP protocol rules enforce that only a Provider may send a `TransferStartMessage`.
If the Consumer attempts to invoke `/dsp/current/transfers/rpc/setup-start`, the request is rejected with a validation error (`only a Provider sends a TransferStartMessage`).

In [15]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


Status Code: 400
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
  "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
  "code": "400",
  "reason": [
    "role: only a Provider sends a TransferStartMessage",
    "state: TransferStartMessage is not allowed from state STARTED"
  ]
}


## 7. Shape Validation Test: Reject Non-RDF / Plain JSON
Eunomia RPC endpoints require valid JSON-LD messages with `@context` and `@type`.
Sending plain JSON without the Eunomia `@context` or matching `@type` is rejected by the RPC shape validator.

In [16]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


Status Code: 400
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "",
  "providerPid": "",
  "code": "400",
  "reason": [
    "body: expanded body declares no message type"
  ]
}


## 8. Terminate Transfer (Consumer -> Provider)
The Consumer terminates the active transfer process by sending a `RpcTransferTerminationMessage` via `/dsp/current/transfers/rpc/setup-termination`.
The transfer process moves to the terminal state `TERMINATED`.

In [17]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-termination"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferTerminationMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "STOPPED",
    "reason": ["Demonstration completed"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferTerminationMessage",
    "code": "STOPPED",
    "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
    "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
    "reason": [
      "Demonstration completed"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:6d46c2ef-44ee-4bef-ba64-ebb662d26e24",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "consumer",
    "protocol": "dsp2025_1",
    "state": "TERMINATED",
    "stateMetadata": {
      "attribute": "BY_CONSUMER",
      "reason": [
        "Demonstration completed"
      ],
      "code": "STOPPED"
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
      "providerPid": "ur

## 9. Query Final Transfer Process State
Query the Provider's Transfer Agent Management API (`/api/v1/transfer-agent/transfer-processes`) to verify the final recorded state and audit history.

In [18]:
url = data_space_provider + "/api/v1/transfer-agent/transfer-processes"
response = requests.request("GET", url, headers=json_header)
print("Processes in Provider:")
print(json.dumps(response.json(), indent=2))


Processes in Provider:
{
  "items": [
    {
      "id": "urn:transfer-process:353c621b-3f5b-4420-aaaf-ed777380f088",
      "userId": "system",
      "userRole": "/admin",
      "visibility": "Public",
      "role": "provider",
      "protocol": "dsp2025_1",
      "state": "TERMINATED",
      "stateMetadata": {
        "attribute": "BY_CONSUMER",
        "reason": [
          "Demonstration completed"
        ],
        "code": "STOPPED"
      },
      "correlation": {
        "identifiers": {
          "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
          "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd"
        },
        "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
        "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
        "agreementId": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
        "callbackAddress": "http://127.0.0.1:1100/dsp/current",
        "peerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0e